In [2]:
import pandas as pd
from math import sqrt

In [3]:
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings_sample.csv')

In [4]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## Preprocessing


In [5]:
movies_df['year'] = movies_df.title.str.extract('(\(\d\d\d\d\))',expand=False)

movies_df['year'] = movies_df.year.str.extract('(\d\d\d\d)',expand=False)

movies_df['title'] = movies_df.title.str.replace('(\(\d\d\d\d\))', '')

movies_df['title'] = movies_df['title'].apply(lambda x: x.strip())

C:\Users\bgc\AppData\Local\Temp/ipykernel_8012/1974451699.py:5: FutureWarning: The default value of regex will change from True to False in a future version.
  movies_df['title'] = movies_df.title.str.replace('(\(\d\d\d\d\))', '')


In [6]:
movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


In [7]:
movies_df=movies_df.drop('genres',1)
movies_df.head()

C:\Users\bgc\AppData\Local\Temp/ipykernel_8012/2395529687.py:1: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only
  movies_df=movies_df.drop('genres',1)


,movieId,title,year
0,1,Toy Story,1995
1,2,Jumanji,1995
2,3,Grumpier Old Men,1995
3,4,Waiting to Exhale,1995
4,5,Father of the Bride Part II,1995


In [8]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,169,2.5,1204927694
1,1,2471,3.0,1204927438
2,1,48516,5.0,1204927435
3,2,2571,3.5,1436165433
4,2,109487,4.0,1436165496


In [9]:
ratings_df = ratings_df.drop('timestamp', 1)

C:\Users\bgc\AppData\Local\Temp/ipykernel_8012/2083535515.py:1: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only
  ratings_df = ratings_df.drop('timestamp', 1)


In [10]:
ratings_df.head()

,userId,movieId,rating
0,1,169,2.5
1,1,2471,3.0
2,1,48516,5.0
3,2,2571,3.5
4,2,109487,4.0


## Collaborative Filtering

In [11]:
userInput = [
            {'title':'Breakfast Club, The', 'rating':5},
            {'title':'Toy Story', 'rating':3.5},
            {'title':'Jumanji', 'rating':2},
            {'title':"Pulp Fiction", 'rating':5},
            {'title':'Akira', 'rating':4.5}
         ] 
inputMovies = pd.DataFrame(userInput)
inputMovies

,title,rating
0,"Breakfast Club, The",5.0
1,Toy Story,3.5
2,Jumanji,2.0
3,Pulp Fiction,5.0
4,Akira,4.5


In [12]:
inputId=movies_df[movies_df['title'].isin(inputMovies['title'].tolist())]

inputMovies=pd.merge(inputId,inputMovies)

inputMovies=inputMovies.drop('year',1)

inputMovies


C:\Users\bgc\AppData\Local\Temp/ipykernel_8012/3589060429.py:5: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only
  inputMovies=inputMovies.drop('year',1)


,movieId,title,rating
0,1,Toy Story,3.5
1,2,Jumanji,2.0
2,296,Pulp Fiction,5.0
3,1274,Akira,4.5
4,1968,"Breakfast Club, The",5.0


#### The users who has seen the same movies

In [13]:
userSubset=ratings_df[ratings_df['movieId'].isin(inputMovies['movieId'].tolist())]
userSubset.head()

,userId,movieId,rating
19,4,296,4.0
441,12,1968,3.0
479,13,2,2.0
531,13,1274,5.0
681,14,296,2.0


In [14]:
userSubset.shape

(33534, 3)

In [15]:
userSubsetGroup=userSubset.groupby(['userId'])

In [16]:
userSubsetGroup.describe()

movieId                                                            \
         count         mean          std     min     25%     50%     75%   
userId                                                                     
4          1.0   296.000000          NaN   296.0   296.0   296.0   296.0   
12         1.0  1968.000000          NaN  1968.0  1968.0  1968.0  1968.0   
13         2.0   638.000000   899.439826     2.0   320.0   638.0   956.0   
14         1.0   296.000000          NaN   296.0   296.0   296.0   296.0   
15         3.0   755.000000  1060.793571     1.0   148.5   296.0  1132.0   
...        ...          ...          ...     ...     ...     ...     ...   
42117      1.0   296.000000          NaN   296.0   296.0   296.0   296.0   
42118      3.0    99.666667   170.030389     1.0     1.5     2.0   149.0   
42121      1.0  1968.000000          NaN  1968.0  1968.0  1968.0  1968.0   
42127      1.0   296.000000          NaN   296.0   296.0   296.0   296.0   
42128      2.0   149.000000   207.889394     2.0    75.5   149.0   222.5   

               rating                                                 
           max  count      mean       std  min   25%  50%   75%  max  
userId                                                                
4        296.0    1.0  4.000000       NaN  4.0  4.00  4.0  4.00  4.0  
12      1968.0    1.0  3.000000       NaN  3.0  3.00  3.0  3.00  3.0  
13      1274.0    2.0  3.500000  2.121320  2.0  2.75  3.5  4.25  5.0  
14       296.0    1.0  2.000000       NaN  2.0  2.00  2.0  2.00  2.0  
15      1968.0    3.0  3.333333  0.577350  3.0  3.00  3.0  3.50  4.0  
...        ...    ...       ...       ...  ...   ...  ...   ...  ...  
42117    296.0    1.0  2.000000       NaN  2.0  2.00  2.0  2.00  2.0  
42118    296.0    3.0  3.333333  0.577350  3.0  3.00  3.0  3.50  4.0  
42121   1968.0    1.0  4.000000       NaN  4.0  4.00  4.0  4.00  4.0  
42127    296.0    1.0  4.500000       NaN  4.5  4.50  4.5  4.50  4.5  
42128    296.0    2.0  3.500000  0.707107  3.0  3.25  3.5  3.75  4.0  

[19795 rows x 16 columns]

In [17]:
userSubsetGroup.get_group(42118)

,userId,movieId,rating
3899036,42118,1,3.0
3899037,42118,2,4.0
3899049,42118,296,3.0


In [18]:
userSubsetGroup.get_group(1130)

,userId,movieId,rating
104167,1130,1,0.5
104168,1130,2,4.0
104214,1130,296,4.0
104363,1130,1274,4.5
104443,1130,1968,4.5


In [19]:
userSubsetGroup=sorted(userSubsetGroup, key=lambda x:len(x[1]), reverse=True)

In [20]:
userSubsetGroup[0:3]

[(75,
        userId  movieId  rating
  7507      75        1     5.0
  7508      75        2     3.5
  7540      75      296     5.0
  7633      75     1274     4.5
  7673      75     1968     5.0),
 (106,
        userId  movieId  rating
  9083     106        1     2.5
  9084     106        2     3.0
  9115     106      296     3.5
  9198     106     1274     3.0
  9238     106     1968     3.5),
 (686,
         userId  movieId  rating
  61336     686        1     4.0
  61337     686        2     3.0
  61377     686      296     4.0
  61478     686     1274     4.0
  61569     686     1968     5.0)]

#### Similarity of users to input user

### select a subset of users to iterate through. This limit is imposed because we don't want to waste too much time going through every single user.

In [21]:
userSubsetGroup=userSubsetGroup[0:100]

In [22]:

pearsonCorrelationDict={}

for name,group in userSubsetGroup:

    group=group.sort_values(by='movieId')
    inputMovies=inputMovies.sort_values(by='movieId')

    nRatings=len(group)

    temp_df=inputMovies[inputMovies['movieId'].isin(group['movieId'].tolist())]

    tempRatingList=temp_df['rating'].tolist()

    tempGroupList=group['rating'].tolist()

    Sxx=sum([i**2 for i in tempRatingList])-pow(sum(tempRatingList),2)/float(nRatings)

    Syy=sum([i**2 for i in tempGroupList])-pow(sum(tempGroupList),2)/float(nRatings)

    Sxy=sum(i*j for i,j in zip (tempRatingList,tempGroupList))-sum(tempRatingList)* sum (tempGroupList)/float(nRatings)

    if Sxx !=0 and Syy !=0:
        pearsonCorrelationDict[name]=Sxy/sqrt(Sxx*Syy)
    else:
        pearsonCorrelationDict[name]=0
  

In [23]:
pearsonCorrelationDict.items()

dict_items([(75, 0.8272781516947562), (106, 0.5860090386731182), (686, 0.8320502943378437), (815, 0.5765566601970551), (1040, 0.9434563530497265), (1130, 0.2891574659831201), (1502, 0.8770580193070299), (1599, 0.4385290096535153), (1625, 0.716114874039432), (1950, 0.179028718509858), (2065, 0.4385290096535153), (2128, 0.5860090386731196), (2432, 0.1386750490563073), (2791, 0.8770580193070299), (2839, 0.8204126541423674), (2948, -0.11720180773462392), (3025, 0.45124262819713973), (3040, 0.89514359254929), (3186, 0.6784622064861935), (3271, 0.26989594817970664), (3429, 0.0), (3734, -0.15041420939904673), (4099, 0.05860090386731196), (4208, 0.29417420270727607), (4282, -0.4385290096535115), (4292, 0.6564386345361464), (4415, -0.11183835382312353), (4586, -0.9024852563942795), (4725, -0.08006407690254357), (4818, 0.4885967564883424), (5104, 0.7674257668936507), (5165, -0.4385290096535153), (5547, 0.17200522903844556), (6082, -0.04728779924109591), (6207, 0.9615384615384616), (6366, 0.65779

In [24]:
pearsonCorrelationDict

{75: 0.8272781516947562,
 106: 0.5860090386731182,
 686: 0.8320502943378437,
 815: 0.5765566601970551,
 1040: 0.9434563530497265,
 1130: 0.2891574659831201,
 1502: 0.8770580193070299,
 1599: 0.4385290096535153,
 1625: 0.716114874039432,
 1950: 0.179028718509858,
 2065: 0.4385290096535153,
 2128: 0.5860090386731196,
 2432: 0.1386750490563073,
 2791: 0.8770580193070299,
 2839: 0.8204126541423674,
 2948: -0.11720180773462392,
 3025: 0.45124262819713973,
 3040: 0.89514359254929,
 3186: 0.6784622064861935,
 3271: 0.26989594817970664,
 3429: 0.0,
 3734: -0.15041420939904673,
 4099: 0.05860090386731196,
 4208: 0.29417420270727607,
 4282: -0.4385290096535115,
 4292: 0.6564386345361464,
 4415: -0.11183835382312353,
 4586: -0.9024852563942795,
 4725: -0.08006407690254357,
 4818: 0.4885967564883424,
 5104: 0.7674257668936507,
 5165: -0.4385290096535153,
 5547: 0.17200522903844556,
 6082: -0.04728779924109591,
 6207: 0.9615384615384616,
 6366: 0.6577935144802716,
 6482: 0.0,
 6530: -0.351605423203

In [25]:
pearsonDF=pd.DataFrame.from_dict(pearsonCorrelationDict,orient='index')
pearsonDF.columns=['similarityIndex']
pearsonDF['userId']=pearsonDF.index
pearsonDF.index=range(len(pearsonDF))
pearsonDF.head()



,similarityIndex,userId
0,0.827278,75
1,0.586009,106
2,0.832050,686
3,0.576557,815
4,0.943456,1040


In [26]:
topUsers=pearsonDF.sort_values(by='similarityIndex',ascending=False)[0:50]
topUsers.head()


,similarityIndex,userId
64,0.961678,12325
34,0.961538,6207
55,0.961538,10707
67,0.960769,13053
4,0.943456,1040


In [27]:
# merge topUser and rating tabel where userIndex are equal
topUsersRating=topUsers.merge(ratings_df, left_on='userId', right_on='userId',how='inner')

In [28]:
topUsersRating.head()

,similarityIndex,userId,movieId,rating
0,0.961678,12325,1,3.5
1,0.961678,12325,2,1.5
2,0.961678,12325,3,3.0
3,0.961678,12325,5,0.5
4,0.961678,12325,6,2.5


In [29]:
topUsersRating.shape

(47240, 4)

In [30]:
# both ways is correct to merge
topUsersRating1=pd.merge(topUsers,ratings_df)

In [31]:
topUsersRating1.shape

(47240, 4)

In [32]:
topUsersRating1.head()

,similarityIndex,userId,movieId,rating
0,0.961678,12325,1,3.5
1,0.961678,12325,2,1.5
2,0.961678,12325,3,3.0
3,0.961678,12325,5,0.5
4,0.961678,12325,6,2.5


In [33]:
topUsersRating['WeightedRating']=topUsersRating['similarityIndex']*topUsersRating['rating']
topUsersRating.head()

,similarityIndex,userId,movieId,rating,WeightedRating
0,0.961678,12325,1,3.5,3.365874
1,0.961678,12325,2,1.5,1.442517
2,0.961678,12325,3,3.0,2.885035
3,0.961678,12325,5,0.5,0.480839
4,0.961678,12325,6,2.5,2.404196


In [34]:
tempTopUserRating=topUsersRating.groupby('movieId').sum()[['similarityIndex','WeightedRating']]
tempTopUserRating.columns=['sum_similarityIndex','sum_WeightedRating']
tempTopUserRating.head()

,sum_similarityIndex,sum_WeightedRating
movieId,,
1,38.376281,140.800834
2,38.376281,96.656745
3,10.253981,27.254477
4,0.929294,2.787882
5,11.723262,27.151751


In [36]:
recommendation_df=pd.DataFrame()
recommendation_df['weighted average recommendation score']=tempTopUserRating['sum_WeightedRating']/tempTopUserRating['sum_similarityIndex']
recommendation_df['movieId']=tempTopUserRating.index
recommendation_df.head()

,weighted average recommendation score,movieId
movieId,,
1,3.668955,1
2,2.518658,2
3,2.657941,3
4,3.000000,4
5,2.316058,5


In [37]:
recommendation_df=recommendation_df.sort_values(by='weighted average recommendation score',ascending=False)
recommendation_df.head(10)

,weighted average recommendation score,movieId
movieId,,
5073,5.0,5073
3329,5.0,3329
2284,5.0,2284
26801,5.0,26801
6776,5.0,6776
6672,5.0,6672
3759,5.0,3759
3769,5.0,3769
3775,5.0,3775


In [38]:
movies_df.loc[movies_df['movieId'].isin(recommendation_df.head(10)['movieId'].tolist())]

,movieId,title,year
2200,2284,Bandit Queen,1994
3243,3329,"Year My Voice Broke, The",1987
3669,3759,Fun and Fancy Free,1947
3679,3769,Thunderbolt and Lightfoot,1974
3685,3775,Make Mine Music,1946
4978,5073,"Son's Room, The (Stanza del figlio, La)",2001
6563,6672,War Photographer,2001
6667,6776,Lagaan: Once Upon a Time in India,2001
9064,26801,Dragon Inn (Sun lung moon hak chan),1992
18106,90531,Shame,2011
